In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
pip install lightgbm wandb transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
"""
============================================================
SMART MCQ SOLVER CHALLENGE — MODEL 1: TF-IDF + LightGBM
============================================================
Classical ML Baseline (Milestone 2 requirement)
Metric: MAP@3 (Mean Average Precision at 3)

Run this cell/script FIRST — no GPU needed.
============================================================
"""

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# ── W&B Setup ──
import wandb
# Use Kaggle Secrets (add WANDB_API_KEY in Add-ons → Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
except Exception:
    wandb.login()  # fallback: interactive prompt

# ── MAP@3 Evaluation ──
def map_at_3(y_true, y_pred_top3):
    """Compute Mean Average Precision @ 3."""
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data Loading ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"  # Kaggle path
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."  # local fallback

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Answer distribution:\n{train['answer'].value_counts()}")

LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Feature Engineering ──
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

def build_tfidf_features(df, prompt_vec=None, option_vecs=None, fit=True):
    """
    Create TF-IDF features by:
    1. TF-IDF on prompts
    2. TF-IDF on each option text
    3. Cosine similarity between prompt and each option (shared vocab)
    4. Text length features
    """
    if fit:
        prompt_vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)
        prompt_tfidf = prompt_vec.fit_transform(df['prompt'].fillna(''))
        option_vecs = {}
        option_tfidfs = {}
        for col in LABELS:
            option_vecs[col] = TfidfVectorizer(max_features=3000, ngram_range=(1,2), sublinear_tf=True)
            option_tfidfs[col] = option_vecs[col].fit_transform(df[col].fillna(''))
        # Shared vectorizer for similarity (same vocab for prompt & options)
        all_text = pd.concat([df['prompt'].fillna('')] + [df[c].fillna('') for c in LABELS])
        shared_vec = TfidfVectorizer(max_features=8000, ngram_range=(1,2), sublinear_tf=True)
        shared_vec.fit(all_text)
        option_vecs['__shared__'] = shared_vec
    else:
        prompt_tfidf = prompt_vec.transform(df['prompt'].fillna(''))
        option_tfidfs = {}
        for col in LABELS:
            option_tfidfs[col] = option_vecs[col].transform(df[col].fillna(''))

    # Length features
    len_feats = np.column_stack([
        df['prompt'].fillna('').str.len().values,
        *[df[c].fillna('').str.len().values for c in LABELS],
        *[df[c].fillna('').str.split().str.len().fillna(0).values for c in LABELS],
    ])

    # Similarity features using SHARED vectorizer (same dimensions for prompt & option)
    shared_vec = option_vecs['__shared__']
    prompt_shared = shared_vec.transform(df['prompt'].fillna(''))
    sim_feats = np.column_stack([
        np.array(prompt_shared.multiply(
            shared_vec.transform(df[c].fillna(''))
        ).sum(axis=1)).flatten()
        for c in LABELS
    ])

    # Combine: all option TF-IDFs + length + similarity
    combined = hstack([prompt_tfidf] + [option_tfidfs[c] for c in LABELS])

    # Add dense features
    dense = np.hstack([len_feats, sim_feats])

    from scipy.sparse import csr_matrix
    combined = hstack([combined, csr_matrix(dense)])

    return combined, prompt_vec, option_vecs

X_train, prompt_vec, option_vecs = build_tfidf_features(train, fit=True)
X_test, _, _ = build_tfidf_features(test, prompt_vec, option_vecs, fit=False)
y_train = train['answer'].map(label2id).values
print(f"Features: X_train={X_train.shape}, X_test={X_test.shape}")

# ── Cross-Validation ──
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))

run = wandb.init(
    project="smart-mcq-solver",
    name="model1-tfidf-lgbm",
    config={"model": "LightGBM", "features": "TF-IDF", "n_folds": N_FOLDS}
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.05, num_leaves=63,
        max_depth=8, subsample=0.8, colsample_bytree=0.5,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42+fold,
        n_jobs=-1, verbose=-1
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )

    val_probs = model.predict_proba(X_val)
    oof_probs[val_idx] = val_probs
    test_probs += model.predict_proba(X_test) / N_FOLDS

    # Fold MAP@3
    val_top3 = [
        [id2label[i] for i in np.argsort(-p)[:3]] for p in val_probs
    ]
    val_true = [id2label[y] for y in y_val]
    fold_map3 = map_at_3(val_true, val_top3)
    print(f"Fold {fold+1} MAP@3: {fold_map3:.4f}")
    wandb.log({"fold": fold+1, "fold_map3": fold_map3})

# Overall OOF MAP@3
oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_train]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 1 (LightGBM) Overall OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

# ── Save Model 1 Predictions ──
np.save("lgbm_test_probs.npy", test_probs)
np.save("lgbm_oof_probs.npy", oof_probs)

# ── Generate Model 1 Submission ──
test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_lgbm.csv", index=False)
print("Model 1 submission saved to submission_lgbm.csv")
print(sub.head(10))


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f1002825 (24f1002825-iit-madras-foundation) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Train: (2000, 8), Test: (500, 7)
Answer distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Features: X_train=(2000, 17964), X_test=(500, 17964)


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260605_143134-13q0cv23
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model1-tfidf-lgbm
wandb: ⭐️ View project at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/13q0cv23



--- Fold 1/5 ---
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.0250038
[200]	valid_0's multi_logloss: 0.0057765
[300]	valid_0's multi_logloss: 0.00410418
[400]	valid_0's multi_logloss: 0.00388996
[500]	valid_0's multi_logloss: 0.00386324
[600]	valid_0's multi_logloss: 0.00385501
[700]	valid_0's multi_logloss: 0.00385183
[800]	valid_0's multi_logloss: 0.00385042
[900]	valid_0's multi_logloss: 0.00384786
[1000]	valid_0's multi_logloss: 0.00384675
Did not meet early stopping. Best iteration is:
[1000]	valid_0's multi_logloss: 0.00384675
Fold 1 MAP@3: 1.0000

--- Fold 2/5 ---
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.0243007
[200]	valid_0's multi_logloss: 0.00645413
[300]	valid_0's multi_logloss: 0.00466549
[400]	valid_0's multi_logloss: 0.00439445
[500]	valid_0's multi_logloss: 0.00436358
[600]	valid_0's multi_logloss: 0.00435029
[700]	valid_0's multi_logloss: 0.00434433
[800]	valid_0's mu

wandb: updating run metadata


Fold 5 MAP@3: 1.0000

=== Model 1 (LightGBM) Overall OOF MAP@3: 1.0000 ===


wandb: 
wandb: Run history:
wandb:         fold ▁▃▅▆█
wandb:    fold_map3 ▁▁▁▁▁
wandb: overall_map3 ▁
wandb: 
wandb: Run summary:
wandb:         fold 5
wandb:    fold_map3 1
wandb: overall_map3 1
wandb: 
wandb: 🚀 View run model1-tfidf-lgbm at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/13q0cv23
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260605_143134-13q0cv23/logs


Model 1 submission saved to submission_lgbm.csv
   ID Prediction
0   1      A C E
1   2      B D C
2   3      B E A
3   4      E C A
4   5      C D A
5   6      D C B
6   7      E C D
7   8      B C A
8   9      C B A
9  10      B D C


In [4]:
"""
============================================================
SMART MCQ SOLVER CHALLENGE — MODEL 2: PyTorch Text CNN
============================================================
Neural Network built FROM SCRATCH (Milestone 3 requirement)
Uses pre-trained GloVe embeddings but custom CNN architecture.
Metric: MAP@3

Requires GPU. Run on Kaggle with GPU accelerator.
============================================================
"""

import pandas as pd
import numpy as np
import os, gc, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
import wandb

# ── Reproducibility ──
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── MAP@3 ──
def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Tokenizer (simple word-level) ──
import re
from collections import Counter

def tokenize(text):
    """Simple whitespace + punctuation tokenizer."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

# Build vocabulary from train + test
all_texts = list(train['prompt']) + [train[c].tolist() for c in LABELS]
all_texts_flat = list(train['prompt']) + list(test['prompt'])
for c in LABELS:
    all_texts_flat += list(train[c]) + list(test[c])

word_counts = Counter()
for text in all_texts_flat:
    word_counts.update(tokenize(str(text)))

# Keep top 50K words
VOCAB_SIZE = 50000
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in word_counts.most_common(VOCAB_SIZE):
    vocab[word] = len(vocab)
print(f"Vocabulary size: {len(vocab)}")

def encode_text(text, max_len=128):
    tokens = tokenize(str(text))[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    # Pad
    ids = ids + [0] * (max_len - len(ids))
    return ids

# ── Load GloVe Embeddings ──
GLOVE_DIM = 100
GLOVE_PATH = "/kaggle/input/glove-global-vectors-for-word-representation/glove.6B.100d.txt"
if not os.path.exists(GLOVE_PATH):
    GLOVE_PATH = "glove.6B.100d.txt"  # local fallback

embedding_matrix = np.random.normal(0, 0.1, (len(vocab), GLOVE_DIM)).astype(np.float32)
embedding_matrix[0] = 0  # PAD = zeros

if os.path.exists(GLOVE_PATH):
    print("Loading GloVe embeddings...")
    with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in vocab:
                embedding_matrix[vocab[word]] = np.array(parts[1:], dtype=np.float32)
    print("GloVe loaded.")
else:
    print("WARNING: GloVe not found, using random embeddings.")
    print("Add 'glove-global-vectors-for-word-representation' as a Kaggle dataset.")

# ── Dataset ──
MAX_PROMPT_LEN = 80
MAX_OPTION_LEN = 100

class MCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = encode_text(row['prompt'], MAX_PROMPT_LEN)
        option_ids = [encode_text(row[c], MAX_OPTION_LEN) for c in LABELS]

        prompt_t = torch.tensor(prompt_ids, dtype=torch.long)
        options_t = torch.tensor(option_ids, dtype=torch.long)  # (5, max_option_len)

        if self.is_test:
            return prompt_t, options_t
        else:
            label = label2id[row['answer']]
            return prompt_t, options_t, torch.tensor(label, dtype=torch.long)

# ── Text CNN Model (from scratch) ──
class TextCNNMCQ(nn.Module):
    """
    Custom Text CNN for Multiple Choice QA.
    Architecture:
    - Shared embedding layer (pre-trained GloVe)
    - 1D CNN with multiple kernel sizes over prompt and each option
    - Concatenate prompt + option representations
    - FC layers to produce a score per option
    """
    def __init__(self, pretrained_embeddings, kernel_sizes=[2, 3, 4, 5],
                 num_filters=128, hidden_dim=256, dropout=0.3):
        super().__init__()
        vocab_size, embed_dim = pretrained_embeddings.shape

        # Embedding (frozen GloVe + trainable fine-tune layer)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(pretrained_embeddings), requires_grad=False
        )
        self.embed_proj = nn.Linear(embed_dim, embed_dim)  # learnable projection

        # Multi-kernel CNN
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k, padding=k//2)
            for k in kernel_sizes
        ])
        self.conv_bn = nn.ModuleList([
            nn.BatchNorm1d(num_filters) for _ in kernel_sizes
        ])

        cnn_out = num_filters * len(kernel_sizes)

        # Classifier: prompt_feats + option_feats → score
        self.fc = nn.Sequential(
            nn.Linear(cnn_out * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def encode_text(self, x):
        """x: (batch, seq_len) → (batch, cnn_out)"""
        emb = self.embedding(x)          # (B, L, D)
        emb = self.embed_proj(emb)       # (B, L, D)
        emb = emb.permute(0, 2, 1)      # (B, D, L)

        conv_outs = []
        for conv, bn in zip(self.convs, self.conv_bn):
            c = F.relu(bn(conv(emb)))    # (B, F, L')
            c = F.adaptive_max_pool1d(c, 1).squeeze(-1)  # (B, F)
            conv_outs.append(c)

        return torch.cat(conv_outs, dim=1)  # (B, cnn_out)

    def forward(self, prompt, options):
        """
        prompt:  (B, prompt_len)
        options: (B, 5, option_len)
        returns: (B, 5) logits
        """
        B = prompt.size(0)
        prompt_feat = self.encode_text(prompt)  # (B, cnn_out)

        logits = []
        for i in range(5):
            opt_feat = self.encode_text(options[:, i, :])  # (B, cnn_out)
            combined = torch.cat([prompt_feat, opt_feat], dim=1)  # (B, 2*cnn_out)
            score = self.fc(combined)  # (B, 1)
            logits.append(score)

        return torch.cat(logits, dim=1)  # (B, 5)

# ── Training ──
N_FOLDS = 5
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3

# Use Kaggle Secrets (add WANDB_API_KEY in Add-ons → Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
except Exception:
    wandb.login()  # fallback: interactive prompt

wandb.init(project="smart-mcq-solver", name="model2-textcnn",
           config={"model": "TextCNN", "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE})

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
y_all = train['answer'].map(label2id).values
oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))
test_ds = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train, y_all)):
    print(f"\n{'='*50}")
    print(f"Fold {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")

    train_ds = MCQDataset(train.iloc[tr_idx])
    val_ds   = MCQDataset(train.iloc[val_idx])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = TextCNNMCQ(embedding_matrix).to(DEVICE)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_map3 = 0
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for prompt, options, labels in train_loader:
            prompt, options, labels = prompt.to(DEVICE), options.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(prompt, options)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        # Validation
        model.eval()
        val_preds_all = []
        val_labels_all = []
        with torch.no_grad():
            for prompt, options, labels in val_loader:
                prompt, options = prompt.to(DEVICE), options.to(DEVICE)
                logits = model(prompt, options)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                val_preds_all.append(probs)
                val_labels_all.append(labels.numpy())

        val_preds_all = np.concatenate(val_preds_all)
        val_labels_all = np.concatenate(val_labels_all)

        val_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_preds_all]
        val_true = [id2label[y] for y in val_labels_all]
        epoch_map3 = map_at_3(val_true, val_top3)

        avg_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val MAP@3: {epoch_map3:.4f}")
        wandb.log({"fold": fold+1, "epoch": epoch+1, "train_loss": avg_loss, "val_map3": epoch_map3})

        if epoch_map3 > best_map3:
            best_map3 = epoch_map3
            best_val_preds = val_preds_all.copy()
            # Save best model state for test inference
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f"Fold {fold+1} Best MAP@3: {best_map3:.4f}")
    oof_probs[val_idx] = best_val_preds

    # Test inference with best model
    model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    fold_test_preds = []
    with torch.no_grad():
        for prompt, options in test_loader:
            prompt, options = prompt.to(DEVICE), options.to(DEVICE)
            logits = model(prompt, options)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            fold_test_preds.append(probs)
    test_probs += np.concatenate(fold_test_preds) / N_FOLDS

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# Overall OOF MAP@3
oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_all]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 2 (TextCNN) Overall OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

np.save("textcnn_test_probs.npy", test_probs)
np.save("textcnn_oof_probs.npy", oof_probs)

# Submission
test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_textcnn.csv", index=False)
print("Model 2 submission saved to submission_textcnn.csv")
print(sub.head(10))


Device: cuda
Vocabulary size: 2975
Add 'glove-global-vectors-for-word-representation' as a Kaggle dataset.


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260605_143235-ikjdu8g0
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model2-textcnn
wandb: ⭐️ View project at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/ikjdu8g0



Fold 1/5
Epoch 1/15 | Loss: 1.5385 | Val MAP@3: 0.9167
Epoch 2/15 | Loss: 0.6110 | Val MAP@3: 0.9962
Epoch 3/15 | Loss: 0.1564 | Val MAP@3: 1.0000
Epoch 4/15 | Loss: 0.1132 | Val MAP@3: 1.0000
Epoch 5/15 | Loss: 0.0715 | Val MAP@3: 1.0000
Epoch 6/15 | Loss: 0.0655 | Val MAP@3: 1.0000
Epoch 7/15 | Loss: 0.0373 | Val MAP@3: 1.0000
Epoch 8/15 | Loss: 0.0446 | Val MAP@3: 1.0000
Epoch 9/15 | Loss: 0.0348 | Val MAP@3: 1.0000
Epoch 10/15 | Loss: 0.0207 | Val MAP@3: 1.0000
Epoch 11/15 | Loss: 0.0061 | Val MAP@3: 1.0000
Epoch 12/15 | Loss: 0.0125 | Val MAP@3: 1.0000
Epoch 13/15 | Loss: 0.0068 | Val MAP@3: 1.0000
Epoch 14/15 | Loss: 0.0043 | Val MAP@3: 1.0000
Epoch 15/15 | Loss: 0.0041 | Val MAP@3: 1.0000
Fold 1 Best MAP@3: 1.0000

Fold 2/5
Epoch 1/15 | Loss: 1.4650 | Val MAP@3: 0.9304
Epoch 2/15 | Loss: 0.5493 | Val MAP@3: 1.0000
Epoch 3/15 | Loss: 0.1627 | Val MAP@3: 1.0000
Epoch 4/15 | Loss: 0.1007 | Val MAP@3: 1.0000
Epoch 5/15 | Loss: 0.0616 | Val MAP@3: 1.0000
Epoch 6/15 | Loss: 0.0673 | 

wandb: updating run metadata



=== Model 2 (TextCNN) Overall OOF MAP@3: 1.0000 ===


wandb: uploading history steps 73-75, summary, console lines 97-101
wandb: 
wandb: Run history:
wandb:        epoch ▂▃▃▄▅▇▇█▁▃▄▅▅▅▇▁▂▃▃▄▆▇▇█▁▂▃▃▅▅▇▁▁▂▃▄▅▆▇█
wandb:         fold ▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆█████
wandb: overall_map3 ▁
wandb:   train_loss █▄▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁█▂▂▂▁▁▁▁█▄▂▂▁▁▁▁
wandb:     val_map3 █████▃█████████▁█████▅████████▁█████████
wandb: 
wandb: Run summary:
wandb:        epoch 15
wandb:         fold 5
wandb: overall_map3 1
wandb:   train_loss 0.01612
wandb:     val_map3 1
wandb: 
wandb: 🚀 View run model2-textcnn at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/ikjdu8g0
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260605_143235-ikjdu8g0/logs


Model 2 submission saved to submission_textcnn.csv
   ID Prediction
0   1      A E B
1   2      B D A
2   3      B E A
3   4      E C D
4   5      C A D
5   6      D E C
6   7      E D C
7   8      B E A
8   9      C E D
9  10      B C E


In [5]:
pip install -U bitsandbytes>=0.46.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [6]:
"""
============================================================
SMART MCQ SOLVER — MODEL 3: LLM Zero-Shot MCQA
============================================================
Uses Qwen2.5-7B-Instruct (4-bit quantized) to answer
knowledge-based MCQs via token probability extraction.

REQUIRES: GPU + Internet ON (to download model first time)
Run: !pip install -q bitsandbytes accelerate
============================================================
"""
# ── Cell 0: Install deps (run this as a SEPARATE cell first) ──
# !pip install -q bitsandbytes accelerate

import pandas as pd
import numpy as np
import os, gc, warnings, torch
import torch.nn.functional as F
warnings.filterwarnings("ignore")

# ── W&B ──
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
except Exception:
    wandb.login()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── MAP@3 ──
def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Load Quantized LLM ──
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_CANDIDATES = [
    "/kaggle/input/qwen2.5-7b-instruct",           # Kaggle dataset (offline)
    "/kaggle/input/qwen2.5/transformers/7b-instruct/1",
    "Qwen/Qwen2.5-7B-Instruct",                     # HuggingFace (online)
]

model_path = None
for p in MODEL_CANDIDATES:
    if os.path.exists(p) or not p.startswith("/"):
        model_path = p
        break
if model_path is None:
    model_path = "Qwen/Qwen2.5-7B-Instruct"
print(f"Using model: {model_path}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.eval()
print("Model loaded successfully!")

# ── Find token IDs for A/B/C/D/E ──
# The model will output " A", " B", etc. after "Answer:"
answer_token_ids = {}
for label in LABELS:
    # Try both with and without space prefix
    ids_space = tokenizer.encode(f" {label}", add_special_tokens=False)
    ids_nospace = tokenizer.encode(label, add_special_tokens=False)
    # Use the last token (handles multi-token cases)
    answer_token_ids[label] = {
        'space': ids_space[-1],
        'nospace': ids_nospace[-1],
    }
print(f"Answer token IDs: {answer_token_ids}")

# ── Inference Function ──
def get_answer_probs(row, strategy="direct"):
    """Get probability distribution over A-E for a single question."""
    prompt_text = str(row['prompt'])
    options = {c: str(row[c]) for c in LABELS}

    if strategy == "direct":
        user_msg = (
            f"Answer the following multiple choice question. "
            f"Reply with ONLY the letter of the correct answer (A, B, C, D, or E).\n\n"
            f"Question: {prompt_text}\n\n"
            f"A) {options['A']}\n"
            f"B) {options['B']}\n"
            f"C) {options['C']}\n"
            f"D) {options['D']}\n"
            f"E) {options['E']}\n\n"
            f"Answer:"
        )
    elif strategy == "cot":
        user_msg = (
            f"Carefully analyze this multiple choice question. "
            f"Think step by step, then give your final answer as a single letter.\n\n"
            f"Question: {prompt_text}\n\n"
            f"A) {options['A']}\n"
            f"B) {options['B']}\n"
            f"C) {options['C']}\n"
            f"D) {options['D']}\n"
            f"E) {options['E']}\n\n"
            f"Let me analyze each option and select the best answer. The correct answer is:"
        )
    else:  # "expert"
        user_msg = (
            f"You are a world-class expert. Select the single best answer.\n\n"
            f"Question: {prompt_text}\n\n"
            f"(A) {options['A']}\n"
            f"(B) {options['B']}\n"
            f"(C) {options['C']}\n"
            f"(D) {options['D']}\n"
            f"(E) {options['E']}\n\n"
            f"The best answer is ("
        )

    # Tokenize
    messages = [{"role": "user", "content": user_msg}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]  # logits for next token

    # Extract probabilities for answer tokens
    probs = np.zeros(5)
    for i, label in enumerate(LABELS):
        # Take max probability across space/nospace variants
        p_space = logits[answer_token_ids[label]['space']].float().item()
        p_nospace = logits[answer_token_ids[label]['nospace']].float().item()
        probs[i] = max(p_space, p_nospace)

    # Softmax to normalize
    probs = np.exp(probs - probs.max())
    probs = probs / probs.sum()
    return probs

# ── Validate on Training Data (subset) ──
print("\n--- Validating on training data (first 100 samples) ---")
val_subset = train.sample(n=min(100, len(train)), random_state=42)
val_probs = []
val_true = []
for idx, row in val_subset.iterrows():
    probs = get_answer_probs(row, strategy="direct")
    val_probs.append(probs)
    val_true.append(row['answer'])
    if len(val_probs) % 20 == 0:
        # Interim score
        top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_probs]
        score = map_at_3(val_true, top3)
        print(f"  Processed {len(val_probs)}/{len(val_subset)}, interim MAP@3: {score:.4f}")

val_probs = np.array(val_probs)
val_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_probs]
val_map3 = map_at_3(val_true, val_top3)
print(f"\n  Validation MAP@3 (direct): {val_map3:.4f}")

# ── Run All 3 Strategies on Full Test Set ──
wandb.init(project="smart-mcq-solver", name="model3-qwen-llm",
           config={"model": model_path, "strategies": ["direct", "cot", "expert"]})

strategies = ["direct", "cot", "expert"]
all_test_probs = {s: [] for s in strategies}

print(f"\n--- Running inference on {len(test)} test questions ---")
for idx, row in test.iterrows():
    for s in strategies:
        probs = get_answer_probs(row, strategy=s)
        all_test_probs[s].append(probs)

    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(test)}")

# Convert to arrays
for s in strategies:
    all_test_probs[s] = np.array(all_test_probs[s])

# ── Also run on FULL training set for OOF & ensemble ──
print(f"\n--- Running inference on {len(train)} train questions ---")
all_train_probs = {s: [] for s in strategies}
for idx, row in train.iterrows():
    for s in strategies:
        probs = get_answer_probs(row, strategy=s)
        all_train_probs[s].append(probs)
    if (idx + 1) % 200 == 0:
        print(f"  Processed {idx + 1}/{len(train)}")

for s in strategies:
    all_train_probs[s] = np.array(all_train_probs[s])

# ── Optimize strategy weights on training data ──
from scipy.optimize import minimize as scipy_minimize

y_train_labels = train['answer'].values
train_strat_arrays = [all_train_probs[s] for s in strategies]

def neg_map3_train(weights):
    w = np.array(weights)
    w = np.abs(w) / np.abs(w).sum()
    blended = sum(wi * arr for wi, arr in zip(w, train_strat_arrays))
    top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in blended]
    return -map_at_3(y_train_labels, top3)

# Grid search
best_score = -1
best_w = [1, 0, 0]
for w1 in np.arange(0, 1.05, 0.1):
    for w2 in np.arange(0, 1.05 - w1, 0.1):
        w3 = 1 - w1 - w2
        if w3 < 0: continue
        score = -neg_map3_train([w1, w2, w3])
        if score > best_score:
            best_score, best_w = score, [w1, w2, w3]

# Refine
result = scipy_minimize(neg_map3_train, best_w, method='Nelder-Mead')
final_w = np.abs(result.x) / np.abs(result.x).sum()
train_map3 = -result.fun

print(f"\nStrategy weights: {dict(zip(strategies, final_w))}")
print(f"Training MAP@3 (multi-strategy ensemble): {train_map3:.4f}")
wandb.log({"train_map3": train_map3,
           "w_direct": final_w[0], "w_cot": final_w[1], "w_expert": final_w[2]})

# ── Blend test predictions ──
test_probs = sum(w * all_test_probs[s] for w, s in zip(final_w, strategies))
oof_probs  = sum(w * all_train_probs[s] for w, s in zip(final_w, strategies))

np.save("llm_test_probs.npy", test_probs)
np.save("llm_oof_probs.npy", oof_probs)

# ── Generate Submission ──
test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_llm.csv", index=False)
print(f"\nModel 3 (LLM) submission saved to submission_llm.csv")
print(sub.head(10))

wandb.finish()

# Cleanup GPU memory
del model
gc.collect()
torch.cuda.empty_cache()


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Device: cuda


Using model: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260605_143720-6gbtitur
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model3-qwen-llm
wandb: ⭐️ View project at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/6gbtitur



--- Running inference on 500 test questions ---
  Processed 50/500
  Processed 100/500
  Processed 150/500
  Processed 200/500
  Processed 250/500
  Processed 300/500
  Processed 350/500
  Processed 400/500
  Processed 450/500
  Processed 500/500

--- Running inference on 2000 train questions ---
  Processed 200/2000
  Processed 400/2000
  Processed 600/2000
  Processed 800/2000
  Processed 1000/2000
  Processed 1200/2000
  Processed 1400/2000
  Processed 1600/2000
  Processed 1800/2000
  Processed 2000/2000


wandb: updating run metadata



Strategy weights: {'direct': np.float64(1.0), 'cot': np.float64(0.0), 'expert': np.float64(0.0)}
Training MAP@3 (multi-strategy ensemble): 0.8885

Model 3 (LLM) submission saved to submission_llm.csv
   ID Prediction
0   1      A E D
1   2      A B D
2   3      C B D
3   4      E A B
4   5      C A D
5   6      A D E
6   7      A E D
7   8      B A D
8   9      A C B
9  10      B D E


wandb: 
wandb: Run history:
wandb: train_map3 ▁
wandb:      w_cot ▁
wandb:   w_direct ▁
wandb:   w_expert ▁
wandb: 
wandb: Run summary:
wandb: train_map3 0.8885
wandb:      w_cot 0
wandb:   w_direct 1
wandb:   w_expert 0
wandb: 
wandb: 🚀 View run model3-qwen-llm at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/6gbtitur
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260605_143720-6gbtitur/logs


In [7]:
"""
============================================================
SMART MCQ SOLVER — ENSEMBLE & FINAL SUBMISSION
============================================================
Combines all available model predictions.
Run AFTER all model scripts have completed.
============================================================
"""
import pandas as pd, numpy as np, os
from scipy.optimize import minimize

def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A','B','C','D','E']
id2label = {i:l for i,l in enumerate(LABELS)}
y_all = train['answer'].values

# ── Load predictions ──
model_oof, model_test = {}, {}
for name, oof_f, test_f in [
    ("lgbm",    "lgbm_oof_probs.npy",    "lgbm_test_probs.npy"),
    ("textcnn", "textcnn_oof_probs.npy",  "textcnn_test_probs.npy"),
    ("deberta", "deberta_oof_probs.npy",  "deberta_test_probs.npy"),
    ("llm",     "llm_oof_probs.npy",      "llm_test_probs.npy"),
]:
    if os.path.exists(oof_f) and os.path.exists(test_f):
        model_oof[name] = np.load(oof_f)
        model_test[name] = np.load(test_f)
        top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in model_oof[name]]
        print(f"  ✓ {name}: OOF MAP@3 = {map_at_3(y_all, top3):.4f}")

if len(model_oof) > 1:
    names = list(model_oof.keys())
    oofs = [model_oof[n] for n in names]
    def neg(w):
        w = np.abs(w); w /= w.sum()
        bl = sum(wi*o for wi,o in zip(w, oofs))
        return -map_at_3(y_all, [[id2label[i] for i in np.argsort(-p)[:3]] for p in bl])
    best_s, best_w = -1, None
    n = len(names)
    from itertools import product
    for combo in product(np.arange(0,1.05,0.1), repeat=n):
        if abs(sum(combo)-1.0) > 0.05: continue
        s = -neg(list(combo))
        if s > best_s: best_s, best_w = s, list(combo)
    r = minimize(neg, best_w, method='Nelder-Mead')
    fw = np.abs(r.x); fw /= fw.sum()
    print(f"\nOptimized: {dict(zip(names, fw.round(4)))}, MAP@3={-r.fun:.4f}")
    ensemble_test = sum(w*model_test[n] for w,n in zip(fw, names))
else:
    name = list(model_test.keys())[0]
    ensemble_test = model_test[name]
    print(f"Single model: {name}")

top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in ensemble_test]
sub = pd.DataFrame({"ID": test["id"], "Prediction": [" ".join(t) for t in top3]})
sub.to_csv("submission.csv", index=False)
print(f"\nFINAL submission.csv saved ({len(sub)} rows)")
print(sub.head(10))
assert all(len(p.split())==3 for p in sub['Prediction']), "Format error!"
print("✓ All checks passed!")


  ✓ lgbm: OOF MAP@3 = 1.0000
  ✓ textcnn: OOF MAP@3 = 1.0000
  ✓ llm: OOF MAP@3 = 0.8885

Optimized: {'lgbm': np.float64(0.0), 'textcnn': np.float64(0.9), 'llm': np.float64(0.1)}, MAP@3=1.0000

FINAL submission.csv saved (500 rows)
   ID Prediction
0   1      A E B
1   2      B A D
2   3      B C E
3   4      E C D
4   5      C A D
5   6      D A E
6   7      E A D
7   8      B A E
8   9      C A E
9  10      B C E
✓ All checks passed!
